# Leak Detection — Version 1: Multi-Model Comparison

**Models compared:** Random Forest, ANN (MLP), KNN, XGBoost

| Setting | Value |
|---|---|
| Split | 75% train / 25% test, stratified |
| Imbalance | `class_weight='balanced'` (RF, ANN) / `scale_pos_weight` (XGBoost) / sample weights (KNN) |
| Scaling | StandardScaler applied to all models |
| Target | Single pipe |

## 0. Colab Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

print('Searching for BattLeDIM files...')
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if '2018_SCADA' in file or 'Leakages' in file:
            print(os.path.join(root, file))

!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib

## 1. Imports & Configuration

In [ ]:
from pathlib import Path
import pandas as pd
import time
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Preprocessing & utils
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('✓ Imports done')

In [ ]:
TARGET_PIPE    = 'p257'
LEAK_THRESHOLD = 0.5
TEST_SIZE      = 0.25
RANDOM_STATE   = 42

DATA_PATH = Path('/content/drive/MyDrive/BattLeDIM')

FILE_PATHS = {
    'demands'  : DATA_PATH / '2018_SCADA_Demands.csv',
    'flows'    : DATA_PATH / '2018_SCADA_Flows.csv',
    'levels'   : DATA_PATH / '2018_SCADA_Levels.csv',
    'pressures': DATA_PATH / '2018_SCADA_Pressures.csv',
    'leakages' : DATA_PATH / '2018_Leakages.csv'
}

OUTPUT_DIR = Path(f'/content/drive/MyDrive/LeakResults/v1_multimodel_{TARGET_PIPE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Target pipe : {TARGET_PIPE}')
print(f'Threshold   : {LEAK_THRESHOLD} L/s')
print(f'Output dir  : {OUTPUT_DIR}')

## 2. Load Data

In [ ]:
def clean_and_load(file_path, delimiter=';', chunk_size=10000):
    print(f'  Loading {file_path.name}...', end=' ')
    chunks = pd.read_csv(file_path, delimiter=delimiter, chunksize=chunk_size)
    chunk_list = []
    for chunk in chunks:
        chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
        for col in chunk.columns:
            if col != 'Timestamp' and chunk[col].dtype == 'object':
                chunk[col] = chunk[col].str.replace(',', '.').astype(float)
        chunk_list.append(chunk)
    df = pd.concat(chunk_list, ignore_index=True)
    print(f'✓  {len(df):,} rows  x  {len(df.columns)} cols')
    return df

print('Loading datasets...')
data = {name: clean_and_load(path) for name, path in FILE_PATHS.items()}
print('\n✓ All loaded')

## 3. Build Feature Matrix & Labels

In [ ]:
feature_df = data['demands'].copy()
for src in ['flows', 'levels', 'pressures']:
    feature_df = feature_df.merge(data[src], on='Timestamp', how='inner')

leak_labels = pd.DataFrame({
    'Timestamp': data['leakages']['Timestamp'],
    'Leak'     : (data['leakages'][TARGET_PIPE] > LEAK_THRESHOLD).astype(int)
})

full_df = feature_df.merge(leak_labels[['Timestamp','Leak']], on='Timestamp', how='inner')
full_df = full_df.sort_values('Timestamp').reset_index(drop=True)

n_leaks = full_df['Leak'].sum()
n_total = len(full_df)
print(f'Dataset shape : {full_df.shape}')
print(f'Leak rows     : {n_leaks:,}  ({100*n_leaks/n_total:.3f}%)')
print(f'Non-leak rows : {n_total-n_leaks:,}  ({100*(n_total-n_leaks)/n_total:.3f}%)')

## 4. Train / Test Split

In [ ]:
X = full_df.drop(['Timestamp', 'Leak'], axis=1)
y = full_df['Leak']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train size  : {len(X_train):,}')
print(f'Test size   : {len(X_test):,}')
print(f'Train leak %: {100*y_train.sum()/len(y_train):.3f}%')
print(f'Test leak % : {100*y_test.sum()/len(y_test):.3f}%')
print('✓ Stratification confirmed')

# Imbalance ratio — needed by XGBoost and KNN
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
imbalance_ratio = n_neg / n_pos
print(f'\nImbalance ratio (neg/pos): {imbalance_ratio:.2f}')

## 5. Feature Scaling

**Why:** ANN and KNN are sensitive to feature scale. StandardScaler brings all features to mean=0, std=1. RF and XGBoost do not require scaling but it does not affect them negatively.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled  = scaler.transform(X_test)        # same scale to test

# Sample weights for KNN — no class_weight param, so we weight rows instead
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print('✓ Scaling done')
print(f'  Mean (should be ~0): {X_train_scaled.mean():.4f}')
print(f'  Std  (should be ~1): {X_train_scaled.std():.4f}')

## 6. Define Models

In [ ]:
# Random Forest — class_weight handles imbalance directly
rf_model = RandomForestClassifier(
    n_estimators     = 100,
    max_depth        = 20,
    min_samples_leaf = 5,
    class_weight     = 'balanced',
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)

# ANN (MLP) — two hidden layers, early stopping prevents overfitting
# class_weight not supported directly in MLPClassifier — imbalance
# handled via sample_weight passed at fit() time
ann_model = MLPClassifier(
    hidden_layer_sizes  = (128, 64),
    activation          = 'relu',
    solver              = 'adam',
    max_iter            = 200,
    early_stopping      = True,
    validation_fraction = 0.1,
    random_state        = RANDOM_STATE
)

# KNN — no class_weight, imbalance handled via sample_weight at fit()
# K=11 is odd (avoids ties) and a reasonable starting point
knn_model = KNeighborsClassifier(
    n_neighbors = 11,
    n_jobs      = -1
)

# XGBoost — scale_pos_weight = neg/pos ratio
xgb_model = XGBClassifier(
    n_estimators     = 100,
    max_depth        = 6,
    learning_rate    = 0.1,
    scale_pos_weight = imbalance_ratio,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    eval_metric      = 'logloss',
    verbosity        = 0
)

print('✓ All models defined')
print(f'  Random Forest : 100 trees, max_depth=20, class_weight=balanced')
print(f'  ANN (MLP)     : layers=(128,64), relu, adam, early_stopping')
print(f'  KNN           : k=11, sample_weight for imbalance')
print(f'  XGBoost       : 100 trees, scale_pos_weight={imbalance_ratio:.2f}')

## 7. Train All Models

In [ ]:
all_results = {}

def train_and_evaluate(name, model, X_tr, X_te, y_tr, y_te,
                        sample_weight=None):
    print(f'\nTraining {name}...')
    t0 = time.time()
    if sample_weight is not None:
        model.fit(X_tr, y_tr, sample_weight=sample_weight)
    else:
        model.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        'model'      : model,
        'y_pred'     : y_pred,
        'cm'         : cm,
        'tn': int(tn), 'fp': int(fp),
        'fn': int(fn), 'tp': int(tp),
        'accuracy'   : round(accuracy_score(y_te, y_pred), 4),
        'precision'  : round(precision_score(y_te, y_pred, zero_division=0), 4),
        'recall'     : round(recall_score(y_te, y_pred, zero_division=0), 4),
        'f1'         : round(f1_score(y_te, y_pred, zero_division=0), 4),
        'train_time' : round(elapsed, 1)
    }

    print(f'  Done in {elapsed:.1f}s')
    print(f'  Accuracy={r["accuracy"]}  Precision={r["precision"]}  Recall={r["recall"]}  F1={r["f1"]}')
    print(f'  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
    return r


all_results['Random Forest'] = train_and_evaluate(
    'Random Forest', rf_model,
    X_train_scaled, X_test_scaled, y_train, y_test
)

all_results['ANN (MLP)'] = train_and_evaluate(
    'ANN (MLP)', ann_model,
    X_train_scaled, X_test_scaled, y_train, y_test,
    sample_weight=sample_weights   # MLP has no class_weight param
)

all_results['KNN'] = train_and_evaluate(
    'KNN', knn_model,
    X_train_scaled, X_test_scaled, y_train, y_test,
    sample_weight=sample_weights
)

all_results['XGBoost'] = train_and_evaluate(
    'XGBoost', xgb_model,
    X_train_scaled, X_test_scaled, y_train, y_test
)

print('\n✓ All models complete')

## 8. Comparison Table

In [ ]:
print('='*65)
print(f'MODEL COMPARISON — {TARGET_PIPE}')
print('='*65)
print(f'{"Model":<16} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Time":>8}')
print('-'*65)
for name, r in all_results.items():
    print(f'{name:<16} {r["accuracy"]:>9.4f} {r["precision"]:>10.4f} '
          f'{r["recall"]:>8.4f} {r["f1"]:>8.4f} {r["train_time"]:>7.1f}s')
print('='*65)
best = max(all_results, key=lambda k: all_results[k]['f1'])
print(f'\nBest model by F1: {best}  ({all_results[best]["f1"]})')

## 9. Confusion Matrices — All Models

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, r) in zip(axes, all_results.items()):
    sns.heatmap(r['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Leak', 'Leak'],
                yticklabels=['No Leak', 'Leak'])
    ax.set_title(f'{name}\nF1={r["f1"]}  Recall={r["recall"]}',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

fig.suptitle(f'Confusion Matrices — {TARGET_PIPE}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved')

## 10. Save Models & Results to Drive

In [ ]:
# Save each model
for name, r in all_results.items():
    fname = name.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    joblib.dump(r['model'], OUTPUT_DIR / f'model_{fname}_{TARGET_PIPE}.pkl')

# Save scaler — needed to preprocess new data with the same scale
joblib.dump(scaler, OUTPUT_DIR / 'scaler.pkl')

# Save metrics JSON
summary = {
    'pipe'            : TARGET_PIPE,
    'leak_threshold'  : LEAK_THRESHOLD,
    'best_model_by_f1': max(all_results, key=lambda k: all_results[k]['f1']),
    'models': {
        name: {
            'accuracy' : r['accuracy'],
            'precision': r['precision'],
            'recall'   : r['recall'],
            'f1'       : r['f1'],
            'tn': r['tn'], 'fp': r['fp'],
            'fn': r['fn'], 'tp': r['tp']
        }
        for name, r in all_results.items()
    }
}
with open(OUTPUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'✓ Saved to: {OUTPUT_DIR}')